In [ ]:
from gym_interface import Agent, State
import copy
import math
import random
import socket
import time
from collections import deque, namedtuple
from typing import Dict, Iterable, List, Literal, Optional, Union
import shutil as shut
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from datetime import datetime
from distutils.util import strtobool
from rlmodel.utils.utils import print_args, print_box, connected_to_internet
import wandb
import setproctitle
from pathlib import Path
import math
import os, sys
from rlmodel.onpolicy.runner.shared.mpe_runner import MPERunner as Runner

*自定义处理函数*

In [ ]:
def parse_Input(action: List[int]) -> str:
    # example:
    if action[0] == -1:
        return ""

    TacticID1 = f'<AgentCMD1><uint64_t>{action[0]}</uint64_t></AgentCMD1>'
    TacticID2 = f'<AgentCMD2><uint64_t>{action[1]}</uint64_t></AgentCMD2>'
    TacticID3 = f'<AgentCMD3><uint64_t>{action[2]}</uint64_t></AgentCMD3>'
    TacticID4 = f'<AgentCMD4><uint64_t>{action[3]}</uint64_t></AgentCMD4>'
    s = '<c>' + TacticID1 + TacticID2 + TacticID3 + TacticID4 +'</c>'

    return s

In [ ]:
def parse_Output(state: Dict[str, any]) -> dict:
    #example:
    tmp = []
    tmp1 = []
    Input = {}
    for input in state:
        for k, v in input.items():
            if k == 'PlaneInfo':
                tmp.append(v)
            elif k == 'TargetMessage':
                tmp1.append(v)
            else:
                Input[k] = v
    Input['PlaneInfo'] = tmp
    Input['TargetMessage'] = tmp1
    
    return Input

In [ ]:
def outputToTensor(state: Dict[str, any], idx: int) -> np.array:
    self = state['PlaneInfo'][idx]['entity_motion_state']
    # enemy = state['TargetMessage'][idx]['entity_motion_state']
    self_pitch = self['AttitudeInfo']['pitch']/10
    self_yaw = self['AttitudeInfo']['yaw']/10
    self_roll = self['AttitudeInfo']['roll']/10
    self_lon = self['PositionInfo']['longitude']/10
    self_lat = self['PositionInfo']['latitude']/10
    self_alt = self['PositionInfo']['altitude']/100
    self_vx = self['ECEFVelocity']['vx']/10
    self_vy = self['ECEFVelocity']['vy']/10
    self_vz = self['ECEFVelocity']['vz']/10
    # enemy_pitch = enemy['AttitudeInfo']['pitch']/10
    # enemy_yaw = enemy['AttitudeInfo']['yaw']/10
    # enemy_roll = enemy['AttitudeInfo']['roll']/10
    # enemy_lon = enemy['PositionInfo']['longitude']/10
    # enemy_lat = enemy['PositionInfo']['latitude']/10
    # enemy_alt = enemy['PositionInfo']['altitude']/100
    # enemy_vx = enemy['ECEFVelocity']['vx']/10
    # enemy_vy = enemy['ECEFVelocity']['vy']/10
    # enemy_vz = enemy['ECEFVelocity']['vz']/10
    # return np.array([self_pitch, self_yaw, self_roll, self_lon, 
    #                  self_lat, self_alt, self_vx, self_vy, self_vz, 
    #                  enemy_pitch, enemy_yaw, enemy_roll, enemy_lon, 
    #                  enemy_lat, enemy_alt, enemy_vx, enemy_vy, enemy_vz])
    return np.array([self_pitch, self_yaw, self_roll, self_lon, 
                     self_lat, self_alt, self_vx, self_vy, self_vz])

In [ ]:
def cal_Reward(state: Dict[str, any], action1 = 0 , action2 = 0, idx=0):
    eps = 1e-6
    # ================= 基础状态读取 =================
    self = state['PlaneInfo'][idx]['entity_motion_state']
    self_state = state['PlaneInfo'][idx]['entity_state']
    if(idx < 2):
        self2 = state['PlaneInfo'][1 - idx]['entity_motion_state']
        enemy1 = state['PlaneInfo'][2]['entity_motion_state']
        enemy2 = state['PlaneInfo'][3]['entity_motion_state']
        
        self2_state = state['PlaneInfo'][1 - idx]['entity_state']
        enemy1_state = state['PlaneInfo'][2]['entity_state']
        enemy2_state = state['PlaneInfo'][3]['entity_state']

    else:
        self2 = state['PlaneInfo'][5 - idx]['entity_motion_state']
        enemy1 = state['PlaneInfo'][0]['entity_motion_state']
        enemy2 = state['PlaneInfo'][1]['entity_motion_state']

        self2_state = state['PlaneInfo'][5 - idx]['entity_state']
        enemy1_state = state['PlaneInfo'][0]['entity_state']
        enemy2_state = state['PlaneInfo'][1]['entity_state']

    self_lon = self['PositionInfo']['longitude']
    self_lat = self['PositionInfo']['latitude']
    self_alt = self['PositionInfo']['altitude']

    self_vx = self['ECEFVelocity']['vx']
    self_vz = self['ECEFVelocity']['vz']

    
    # ================= 相对位置（米） =================
    def rel_pos(enemy):
        x = (enemy['PositionInfo']['latitude'] - self_lat) * 111000.0
        y = (enemy['PositionInfo']['altitude'] - self_alt)
        z = (enemy['PositionInfo']['longitude'] - self_lon) * 111000.0 * math.cos(self_lat * math.pi / 180)
        return x, y, z

    x1, y1, z1 = rel_pos(enemy1)
    x2, y2, z2 = rel_pos(enemy2)

    d1 = math.sqrt(x1*x1 + y1*y1 + z1*z1) + eps
    d2 = math.sqrt(x2*x2 + y2*y2 + z2*z2) + eps

    # ================= 距离奖励（★修改：减缓衰减） =================
    d0 = 8000.0          # ★ 原来 5000 太激进
    R_dist = max(
        math.exp(-d1 / d0),
        math.exp(-d2 / d0)
    )                     # ∈ (0,1]

    # ================= 偏离角 φ（★修改：加平滑） =================
    def cos_phi(x, z):
        denom = (math.sqrt(self_vx**2 + self_vz**2) *
                 math.sqrt(x**2 + z**2)) + eps
        c = (self_vx * x + self_vz * z) / denom
        c = max(-1.0, min(1.0, c))     # ★ 防 acos 数值抖动
        phi = math.acos(c)
        return max(0.1, math.cos(phi)) # ★ 不再允许为 0

    R_phi = max(cos_phi(x1, z1), cos_phi(x2, z2))

    # ================= 进入角 ψ（★同样平滑） =================
    def cos_psi(enemy, x, z):
        vx = enemy['ECEFVelocity']['vx']
        vz = enemy['ECEFVelocity']['vz']
        denom = (math.sqrt(vx*vx + vz*vz) *
                 math.sqrt(x*x + z*z)) + eps
        c = -(vx * x + vz * z) / denom
        c = max(-1.0, min(1.0, c))
        psi = math.acos(c)
        return max(0.1, math.cos(math.pi - psi))  # ★ 不为 0

    R_psi = max(
        cos_psi(enemy1, x1, z1),
        cos_psi(enemy2, x2, z2)
    )

    # ================= 角度联合奖励 =================
    R_angle = 0.6 * R_psi + 0.4 * R_phi   # ∈ [0.1, 1]

    # ================= 几何奖励（★改为加法而非乘法） =================
    r_geom = 0.5 * R_dist + 0.5 * R_angle   # ★ 避免 double-zero

    # ================= 终止奖励（★去掉突刺） =================
    r_terminal = 0.0
    if enemy1_state == 5 and enemy2_state == 5:
        r_terminal += 0.5
    if self_state == 5 and self2_state == 5:
        r_terminal -= 0.5

    # ================= 最终奖励（★tanh 压缩） =================
    r = r_geom + r_terminal
    r = math.tanh(r)          # ★ PPO 友好
    r = float(r)

    # ================= 安全断言（强烈建议保留） =================
    if not math.isfinite(r):
        r = 0.0
    if(action1 == 2):
        r += 0.03
    if(action2  == 2):
        r += 0.03
    return 10*r


In [ ]:
# global filecnt
# filecnt = 0
# def cal_Termination(state:Dict[str, any]) -> bool:
#     #example
       
#         self1 = state['PlaneInfo'][0]['entity_state']
#         self2 = state['PlaneInfo'][1]['entity_state']
#         enemy1 = state['PlaneInfo'][2]['entity_state']
#         enemy2 = state['PlaneInfo'][3]['entity_state']
#         if enemy1 == 5 and enemy2 == 5:
#             # dst = (
#             #     "D:\\postgraduate\\FinalProject\\simplecq-master\\images\\2v2\\"
#             #     + datetime.now().strftime("%Y%m%d-%H%M%S")
#             #     + "\\data"
#             # )
#             dst = (
#                 "D:\\postgraduate\\FinalProject\\simplecq-master\\images\\2v2\\"
#                 + {filecnt}
#                 + "\\data"
#             )
#             if not os.path.exists(dst):
#                 shut.copytree(
#                     "D:\\postgraduate\\FinalProject\\simplecq-master\\data",
#                     dst
#                 )
#                 filecnt += 1
#             return True
#         elif self1 == 5 and self2 == 5:
#             return True
#         else:    
#             return False
filecnt = 0

def cal_Termination(state: Dict[str, any]) -> bool:
    global filecnt

    self1 = state['PlaneInfo'][0]['entity_state']
    self2 = state['PlaneInfo'][1]['entity_state']
    enemy1 = state['PlaneInfo'][2]['entity_state']
    enemy2 = state['PlaneInfo'][3]['entity_state']

    if enemy1 == 5 and enemy2 == 5:

        # dst = (
        #     "D:\\postgraduate\\FinalProject\\simplecq-master\\images\\2v2\\"
        #     + str(filecnt)
        #     + "\\data"
        # )

        # if not os.path.exists(dst):
        #     shut.copytree(
        #         "D:\\postgraduate\\FinalProject\\simplecq-master\\data",
        #         dst
        #     )
        #     filecnt += 1

        return True

    elif self1 == 5 and self2 == 5:
        return True

    else:
        return False

_算法参数配置_

In [ ]:
device = torch.device("cuda")
sys.path.append(os.path.abspath(os.getcwd()))
num_agents = 2
num_enemies = 2
episode_length = 20
save_interval = 500
log_interval = 10
model_dir = (
        Path(os.path.dirname(os.path.dirname(os.getcwd())) + "/results")
        /"save"
    )
all_args = {
    "algorithm_name": "rmappo",
    "use_recurrent_policy": True,
    "use_naive_recurrent_policy": False,
    "share_policy": True,
    "use_wandb": True,
    "seed": 0,
    "use_centralized_V": True,
    "use_linear_lr_decay": True,
    "hidden_size": 32,
    "recurrent_N": 1,
    "act_space": 5,
    "obs_space": 9,
    "shared_obs_space": 9*num_agents,
    "model_dir": None,
    "episode_length": episode_length,
    "gamma": 0.98,
    "gae_lambda": 0.95,
    "use_gae": True,
    "clip_param": 0.2,
    "ppo_epoch": 15,
    "num_mini_batch": 1,
    "data_chunk_length": 10,
    "value_loss_coef": 0.5,
    "entropy_coef": 0.01,
    "max_grad_norm": 10.0,
    "huber_delta": 10.0,
    "use_max_grad_norm": True,
    "use_clipped_value_loss": True,
    "use_huber_loss": True,
    "use_popart": True,
    "use_valuenorm": False,
    "use_value_active_masks": True,
    "use_policy_active_masks": True,
    "lr": 7e-5,
    "critic_lr": 7e-4,
    "opti_eps": 1e-5,
    "weight_decay": 0,
    "gain": 0.01,
    "use_orthogonal": True,
    "use_feature_normalization": True,
    "use_ReLU": False,
    "stacked_frames": 1,
    "layer_N": 1,
    "n_rollout_threads": 1,
}


run_dir = (
        Path(os.path.dirname(os.path.dirname(os.getcwd())) + "/results")
        / all_args["algorithm_name"]
    )
config = {
    "all_args": all_args,
    "num_agents": num_agents,
    "num_enemies":num_enemies,
    "device": device,
    "run_dir": run_dir
}

*W&B记录训练日志*

In [ ]:
if all_args["use_wandb"]:
        # for supercloud when no internet_connection
        if not connected_to_internet():
            import json

            # save a json file with your wandb api key in your
            # home folder as {'my_wandb_api_key': 'INSERT API HERE'}
            # NOTE this is only for running on systems without internet access
            # have to run `wandb sync wandb/run_name` to sync logs to wandboard
            with open(os.path.dirname(os.path.dirname(os.getcwd())) + "/keys.json") as json_file:
                key = json.load(json_file)
                my_wandb_api_key = key["my_wandb_api_key"]  # NOTE change here as well
            os.environ["WANDB_API_KEY"] = my_wandb_api_key
            os.environ["WANDB_MODE"] = "dryrun"
            os.environ["WANDB_SAVE_CODE"] = "true"

        print_box("Creating wandboard...")
        run = wandb.init(
            config=all_args,
            project="simplecq",
            # project=all_args.env_name,
            # entity="cc",
            notes=socket.gethostname(),
            name=str(all_args["algorithm_name"])
            + "_2v2_self_learn"
            ,
            # group=all_args.scenario_name,
            dir=str(run_dir),
            # job_type="training",
            reinit=True,
        )
        
setproctitle.setproctitle(
        str(all_args["algorithm_name"])
        + "@"
        + str("lapluma030")
    )

# seed
torch.manual_seed(all_args["seed"])
torch.cuda.manual_seed_all(all_args["seed"])
np.random.seed(all_args["seed"])

In [ ]:
runner_red = Runner(config,"red")
runner_blue = Runner(config,"blue")
policy_actor_state_dict = torch.load(
            str(model_dir) + "/actor1.pt", map_location=torch.device("cpu")
        )
policy_critic_state_dict = torch.load(
                str(model_dir) + "/critic1.pt", map_location=torch.device("cpu")
            )
runner_blue.policy.actor.load_state_dict(policy_actor_state_dict)
runner_blue.policy.critic.load_state_dict(policy_critic_state_dict, strict=False)
train_infos_blue={}

*端口及输出类型指定*

In [ ]:
port = 40029
# outputs_type = "uint64_t"
outputs_type = {
    "AgentCMD1": ["uint64_t"],
    "AgentCMD2": ["uint64_t"]
}

In [ ]:
agent = Agent(port=port, 
              outputs_type=outputs_type,
              process_input=parse_Input,
              process_output=parse_Output,
              reward_func=cal_Reward,
              end_func=cal_Termination)


*训练主循环*

In [ ]:
class RewardEntropyBonus1:
    def __init__(self, buffer_size=1000, num_bins=50, eps=1e-8):
        self.buffer = deque(maxlen=buffer_size)
        self.num_bins = num_bins
        self.eps = eps

    def update(self, reward):
        self.buffer.append(reward)

    def compute(self, reward):
        if len(self.buffer) < 20:
            return 0.0

        rewards = np.array(self.buffer)
        hist, bin_edges = np.histogram(
            rewards, bins=self.num_bins, density=True
        )

        bin_width = bin_edges[1] - bin_edges[0]
        probs = hist * bin_width

        idx = np.searchsorted(bin_edges, reward) - 1
        idx = np.clip(idx, 0, len(probs) - 1)

        return -np.log(probs[idx] + self.eps)

def train(env: Agent, episodes, enable_log=True):
    Win = 0
    Lose = 0
    Tie = 0
    # 时间间隔
    time_interval = 500
    init_val = env.reset()
    print("init_val:", init_val)
    # Reward entropy
    # reward_entropy = RewardEntropyBonus1()

    s = np.expand_dims(np.array([np.zeros((all_args["obs_space"], )) for _ in range(num_agents)]),axis=0)
    r = np.expand_dims(np.array([ [0] for _ in range(num_agents) ]),axis=0)
    dones =  np.expand_dims(np.array([ False for _ in range(num_agents) ]),axis=0)
    # replay buffer
    if runner_red.use_centralized_V:
        share_obs = s.reshape(all_args["n_rollout_threads"], -1)
        share_obs = np.expand_dims(share_obs, 1).repeat(runner_red.num_agents, axis=1)
    else:
        share_obs = s

    runner_red.buffer.share_obs[0] = share_obs.copy()
    runner_red.buffer.obs[0] = s.copy()
    runner_blue.buffer.share_obs[0] = share_obs.copy()
    runner_blue.buffer.obs[0] = s.copy()

    def get_phase_by_episode(ep):
        if ep < 200:
            return 0
        elif ep < 500:
            return 1
        else:
            return 2
        
    global count
    count = 0
    for episode in range(episodes):
        Win = 0
        Lose = 0
        Tie = 0
        phase = get_phase_by_episode(episode)
        runner_red.selfplay_phase = phase
        runner_blue.selfplay_phase = phase
        if all_args["use_linear_lr_decay"]:

            runner_red.trainer.policy.lr_decay(episode, episodes)
            # runner_blue.trainer.policy.lr_decay(episode, episodes)

        for step in range(episode_length*time_interval):
            # Sample actions
            if (step % (time_interval)) == 0:
                (
                    values0,
                    actions0,
                    action_log_probs0,
                    rnn_states0,
                    rnn_states_critic0,
                    actions_env0,
                ) = runner_red.collect(step//time_interval)
                (
                    values1,
                    actions1,
                    action_log_probs1,
                    rnn_states1,
                    rnn_states_critic1,
                    actions_env1,
                ) = runner_blue.collect(step//time_interval)
                TacticList = [1, 2, 10, 14, 27]
                # TacticList = [1, 2, 4, 10, 13, 14, 27, 32]
                outputs = np.array(TacticList)
                obs, rew, done, _ = env.step(action=[outputs[actions0[0,0,0]], outputs[actions0[0,1,0]],
                                                     outputs[actions1[0,0,0]], outputs[actions1[0,1,0]]])
            
                s0 = np.expand_dims(np.array([outputToTensor(obs, i) for i in range(num_agents)]),axis=0)
                r0 = np.expand_dims(np.array([ [cal_Reward(obs,outputs[actions0[0,0,0]],outputs[actions0[0,1,0]],i)] for i in range(num_agents) ]),axis=0)
                dones =  np.expand_dims(np.array([ done for _ in range(num_agents) ]),axis=0)
                s1 = np.expand_dims(np.array([outputToTensor(obs, i+2) for i in range(num_agents)]),axis=0)
                r1 = np.expand_dims(np.array([ [cal_Reward(obs,outputs[actions1[0,0,0]],outputs[actions1[0,1,0]], i+2)] for i in range(num_agents) ]),axis=0)
                # ===== Reward Entropy =====
                # reward_entropy.update(r0)
                # entropy_r = reward_entropy.compute(r0)
                # final_reward = (
                #     (1 - 0.2) * r0
                #     + 0.2 * (entropy_r)
                # )
                data0 = (
                    s0,
                    r0,
                    dones,
                    values0,
                    actions0,
                    action_log_probs0,
                    rnn_states0,
                    rnn_states_critic0,
                )
                data1 = (
                    s1,
                    r1,
                    dones,
                    values1,
                    actions1,
                    action_log_probs1,
                    rnn_states1,
                    rnn_states_critic1,
                )
                # insert data into buffer
                runner_red.insert(data0)
                runner_blue.insert(data1)
                if (done == True) or step >= episode_length*time_interval - 1:
                    beat = beaten = 0
                    if obs['PlaneInfo'][0]['entity_state'] == 5:
                        beaten += 1
                    if obs['PlaneInfo'][1]['entity_state'] == 5:
                        beaten += 1
                    if obs['PlaneInfo'][2]['entity_state'] == 5:
                        beat += 1
                    if obs['PlaneInfo'][3]['entity_state'] == 5:
                        beat += 1

                    if beat > beaten:
                        Win += 1
                    elif beat < beaten: 
                        Lose += 1  
                    else:
                        Tie += 1
                    
                    env.reset()
            else:
                obs, rew, done, _ = env.step(action=[outputs[actions0[0,0,0]], outputs[actions0[0,1,0]],
                                                     outputs[actions1[0,0,0]], outputs[actions1[0,1,0]]])
                if (done == True) or step >= episode_length*time_interval - 1:
                    beat = beaten = 0
                    if obs['PlaneInfo'][0]['entity_state'] == 5:
                        beaten += 1
                    if obs['PlaneInfo'][1]['entity_state'] == 5:
                        beaten += 1
                    if obs['PlaneInfo'][2]['entity_state'] == 5:
                        beat += 1
                    if obs['PlaneInfo'][3]['entity_state'] == 5:
                        beat += 1

                    if beat > beaten:
                        Win += 1
                    elif beat < beaten: 
                        Lose += 1  
                    else:
                        Tie += 1
                    
                    env.reset()
            
            if (step % (time_interval)) == 0:
                runner_red.compute()
                train_infos_red = runner_red.train()
                # runner_blue.compute()
                # train_infos_blue = runner_blue.train()

        # post process
        total_num_steps = (episode + 1) * episode_length

        # save model
        if episode % save_interval == 0 or episode == episodes - 1:
            runner_red.save()
            runner_blue.save()
            if episode != 0:
                red_path = Path(model_dir) / "red"
                blue_path = Path(model_dir) / "blue"
                policy_actor_state_dict = torch.load(
                    red_path / "actor.pt", map_location=torch.device("cpu")
                )
                policy_critic_state_dict = torch.load(
                    red_path / "critic.pt", map_location=torch.device("cpu")
                )
                runner_blue.policy.actor.load_state_dict(policy_actor_state_dict)
                runner_blue.policy.critic.load_state_dict(policy_critic_state_dict, strict=False)
                
                policy_actor_state_dict = torch.load(
                    blue_path / "actor.pt", map_location=torch.device("cpu")
                )
                policy_critic_state_dict = torch.load(
                    blue_path / "critic.pt", map_location=torch.device("cpu")
                )
                runner_red.policy.actor.load_state_dict(policy_actor_state_dict)
                runner_red.policy.critic.load_state_dict(policy_critic_state_dict, strict=False)
        # log information
        if episode % log_interval == 0:

            avg_ep_rew_red = np.mean(runner_red.buffer.rewards) * episode_length
            train_infos_red["average_episode_rewards"] = avg_ep_rew_red
            if Win+Lose != 0 :
                train_infos_red["win_rate"] = (Win/(Win+Lose)) * 100.0
                train_infos_blue["win_rate"] = (Lose/(Win+Lose)) * 100.0
            else :
                train_infos_red["win_rate"] = (1/2) * 100.0
                train_infos_blue["win_rate"] = (1/2) * 100.0
            avg_ep_rew_blue = np.mean(runner_blue.buffer.rewards) * episode_length
            train_infos_blue["average_episode_rewards"] = avg_ep_rew_blue
            print(
                f"Average episode rewards is {avg_ep_rew_red:.3f} \t"
                f"Total timesteps: {total_num_steps} \t "
                f"Percentage complete {total_num_steps / episodes / episode_length * 100:.3f}"
            )
            print(f"Training. Win:{Win} Lose: {Lose} Tie: {Tie}")
            runner_red.log_train_self_learn(train_infos_red, total_num_steps)
            runner_blue.log_train_self_learn(train_infos_blue, total_num_steps)
        
        if episode % 100 == 0 and enable_log:
            print(f"finished: {episode / episodes * 100}%")

    print(f"Training is done. Win:{Win} Lose: {Lose} Tie: {Tie}")
    
    if all_args["use_wandb"]:

        run.finish()

*运行训练*

In [ ]:
train(agent, 1499)
